# HealthGraph — ETL
Matrizes curriculares da UFF (PDF) → pandas → Neo4j.

| Etapa | O que faz |
|---|---|
| **0. Modelagem** | Do relacional ao grafo, convenções de nomes |
| **1. Extract** | Ler os PDFs de `assets/matrizes/` |
| **2. Transform** | Uma tabela pandas por elemento do grafo |
| **3. Load** | Schema (`cypher/schema.cypher`) + `MERGE` com parâmetros |
| **4. Validação** | Contagens e consultas (`cypher/queries.cypher`) |

## 0. Modelagem

| Relacional / ER | Grafo (Neo4j) |
|---|---|
| Tabela / entidade | Label de nó |
| Linha (instância) | Nó |
| Coluna | Propriedade |
| Chave estrangeira | Relacionamento |
| Tabela associativa (N:N) | Relacionamento, com os atributos da associação como propriedades |
| Chave primária | Propriedade identificadora com constraint de unicidade |

**Modelo:**
```
(:Curso {codigo, nome, chObrigatoria, periodos})
   -[:OFERECE {periodo, tipo, chMinima, periodosCompletos}]->
(:Disciplina {codigo, nome, nomeBase, cht, chp, ches, chTotal, chex, desativada})
   -[:REQUER {tipo: 'pre' | 'co'}]->      (:Disciplina)
   -[:EQUIVALE_A]->                       (:Disciplina)   // só se existir data/equivalencias.csv
   -[:DO_DEPTO]->                         (:Departamento {sigla})
```
- só disciplinas obrigatórias (`OB`)

## 1. Extract

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
import pdfplumber
from neo4j import GraphDatabase

NEO4J_URI  = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_AUTH = (os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "neo4j12345"))
NEO4J_DB   = os.getenv("NEO4J_DATABASE", "neo4j")

PDF_DIR    = Path("assets/matrizes")
CYPHER_DIR = Path("cypher")
DATA_DIR   = Path("data")
OUT_DIR    = DATA_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
driver.verify_connectivity()

In [2]:
# colunas da tabela do pdf, na ordem
COLS = ["codigo", "nome", "tipo", "cht", "chp", "ches", "chtotal", "chex", "chprereq", "prereq", "coreq"]
RE_PERIODO = re.compile(r"^(\d+)º período")
RE_CODIGO  = re.compile(r"[A-Z]{3}\d{5}")

In [3]:
# junta as palavras de uma célula na ordem de leitura (linha por linha)
def texto_celula(palavras):
    linhas = []
    for w in sorted(palavras, key=lambda w: w["top"]):
        if linhas and abs(w["top"] - linhas[-1][0]["top"]) < 3:
            linhas[-1].append(w)
        else:
            linhas.append([w])
    return " ".join(" ".join(w["text"] for w in sorted(l, key=lambda w: w["x0"])) for l in linhas)

In [4]:
# cabeçalho do curso (primeira página)
def ler_cabecalho(pdf):
    txt = pdf.pages[0].extract_text()
    return {
        "codigo": re.search(r"Currículo:\s*([\d.]+)", txt).group(1),
        "nome": re.search(r"Curso:\s*(.+)", txt).group(1).strip(),
        "chObrigatoria": int(re.search(r"Carga horária obrigatória:\s*(\d+)", txt).group(1)),
    }

In [5]:
# cada linha da tabela é a faixa entre as linhas verticais da borda esquerda
# as colunas são as posições x das linhas verticais
def ler_linhas(pdf):
    linhas, periodo = [], None
    for pg in pdf.pages:
        verticais = [l for l in pg.lines if abs(l["x0"] - l["x1"]) < 1]
        if not verticais:
            continue
        xmin   = min(l["x0"] for l in verticais)
        xs     = sorted({round(l["x0"]) for l in verticais})
        faixas = sorted({(l["top"], l["bottom"]) for l in verticais if abs(l["x0"] - xmin) < 1})

        # títulos "1º período" ... e "Não periodizada" (optativas)
        marcas = []
        for l in pg.extract_text_lines():
            m = RE_PERIODO.match(l["text"])
            if m:
                marcas.append((l["top"], int(m.group(1))))
            elif l["text"].startswith("Não periodizada"):
                marcas.append((l["top"], None))

        palavras = pg.extract_words()
        for top, bottom in faixas:
            for t, p in marcas:
                if t < top:
                    periodo = p

            celulas = [[] for _ in COLS]
            for w in palavras:
                yc, xc = (w["top"] + w["bottom"]) / 2, (w["x0"] + w["x1"]) / 2
                if top <= yc <= bottom:
                    i = sum(1 for x in xs if x < xc) - 1
                    if 0 <= i < len(COLS):
                        celulas[i].append(w)
            celulas = [texto_celula(c) for c in celulas]

            if celulas[0] == "Código":
                continue
            # linha quebrada entre páginas: o resto vem sem código
            if celulas[0] == "" and linhas:
                for i in (1, 9, 10):
                    if celulas[i]:
                        linhas[-1][COLS[i]] = (linhas[-1][COLS[i]] + " " + celulas[i]).strip()
                continue
            if not RE_CODIGO.fullmatch(celulas[0]):
                continue
            linhas.append(dict(zip(COLS, celulas), periodo=periodo))
    return linhas

In [6]:
# lê todos os pdfs
cursos, brutas = [], []
for arq in sorted(PDF_DIR.glob("*.pdf")):
    with pdfplumber.open(arq) as pdf:
        cab = ler_cabecalho(pdf)
        for l in ler_linhas(pdf):
            l["curso"] = cab["codigo"]
            brutas.append(l)
    cursos.append(cab)
    print(cab["nome"], "ok")

df_bruto = pd.DataFrame(brutas)
df_bruto.head()

BIOMEDICINA ok
ENFERMAGEM ok
FARMÁCIA ok
MEDICINA ok
MEDICINA VETERINÁRIA ok
ODONTOLOGIA ok


,codigo,nome,tipo,cht,chp,ches,chtotal,chex,chprereq,prereq,coreq,periodo,curso
0,GAN00144,COMPLEMENTOS DE MATEMÁTICA APLICADA,OB,60,0,0,60,0,,,,1.0,48.01.002
1,GCM00016,BIOLOGIA CELULAR E MOLECULAR,OB,60,20,0,80,0,,,,1.0,48.01.002
2,GFI00110,FUNDAMENTOS DE FISICA PARA BIOLOGIA,OB,80,0,0,80,0,,,,1.0,48.01.002
3,GQI00050,QUÍMICA GERAL II,OB,60,0,0,60,0,,,,1.0,48.01.002
4,MGB00002,FUNDAMENTOS DE INICIACAO A PESQUISA,OB,40,0,0,40,10,,,,1.0,48.01.002


In [7]:
# quantas linhas de cada tipo vieram de cada curso
df_bruto.pivot_table(index="curso", columns="tipo", values="codigo", aggfunc="count", fill_value=0)

tipo,AC,E,O,OB
curso,,,,
15.05.003,2,0,72,74
16.01.004,14,76,85,65
17.01.003,2,0,46,62
18.01.004,3,0,110,70
34.02.003,6,0,47,53
48.01.002,2,0,141,33


## 2. Transform

In [8]:
# só obrigatórias
df = df_bruto[df_bruto["tipo"] == "OB"].copy()

# números
for c in ["cht", "chp", "ches", "chtotal", "chex"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
df["chprereq"] = pd.to_numeric(df["chprereq"], errors="coerce")
df["periodo"]  = df["periodo"].astype(int)

In [9]:
# padroniza nomes: tira "(Desativada)", espaços duplos, vira flag
df["desativada"] = df["nome"].str.contains("Desativada", case=False)
df["nome"] = (df["nome"]
              .str.replace(r"\(Desativada\)", "", regex=True, case=False)
              .str.replace(r"\s+", " ", regex=True)
              .str.strip()
              .str.upper())

# nome base sem numeral romano no final (ANATOMIA I -> ANATOMIA, FISIOLOGIA VII - A -> FISIOLOGIA)
ROMANO = r"\s+(?:X{0,3})(?:IX|IV|V?I{0,3})(?:\s*-\s*[A-Z])?$"
df["nomeBase"] = df["nome"].str.replace(ROMANO, "", regex=True).str.strip()

df[["codigo", "nome", "nomeBase", "desativada"]].sample(5, random_state=1)

,codigo,nome,nomeBase,desativada
718,MSV00014,DOENCAS INFECCIOSAS DOS ANIM. DOMESTICOS,DOENCAS INFECCIOSAS DOS ANIM. DOMESTICOS,False
679,MMO00084,ANATOMIA ANIMAL II,ANATOMIA ANIMAL,False
891,MOC00086,PERIODONTIA II - A,PERIODONTIA,False
435,MGM00411,PRIMEIROS SOCORROS,PRIMEIROS SOCORROS,False
879,MOC00082,RADIOLOGIA ODONTOLÓGICA E IMAGINOLOGIA II,RADIOLOGIA ODONTOLÓGICA E IMAGINOLOGIA,False


In [10]:
# separa pré-requisitos compostos: cada "[n - COD] NOME" vira uma linha
# pega só o código (o nome vem da própria disciplina)
df["preCodigos"] = df["prereq"].str.findall(r"([A-Z]{3}\d{5})\]")
df["coCodigos"]  = df["coreq"].str.findall(r"([A-Z]{3}\d{5})\]")

# requisito de períodos completos, ex: "7º período completo"
df["periodosCompletos"] = df["prereq"].str.findall(r"(\d+)º período completo").apply(lambda l: [int(x) for x in l])

df[df["preCodigos"].str.len() > 1][["codigo", "prereq", "preCodigos"]].head()

,codigo,prereq,preCodigos
12,MFL00050,[2 - MGB00001] NEUROCIENCIAS [2 - MMO00038] MO...,"[MGB00001, MMO00038]"
24,MPT00063,[2 - GCM00017] BIOQUÍMICA I [3 - MPT00058] PAT...,"[GCM00017, MPT00058]"
25,MPT00064,[1 - GCM00016] BIOLOGIA CELULAR E MOLECULAR [2...,"[GCM00016, GCM00017, MFL00050, MPT00058]"
26,MPT00065,[3 - GIM00021] IMUNOLOGIA V [4 - MIP00045] VIR...,"[GIM00021, MIP00045]"
27,MPT00066,[4 - MIP00042] BACTERIOLOGIA V [4 - MIP00043] ...,"[MIP00042, MIP00043]"


In [11]:
# departamento = prefixo do código
df["departamento"] = df["codigo"].str[:3]

In [12]:
# nós: Curso
nos_curso = pd.DataFrame(cursos)
periodos = df.groupby("curso")["periodo"].max().rename("periodos")
nos_curso = nos_curso.merge(periodos, left_on="codigo", right_index=True)
nos_curso

,codigo,nome,chObrigatoria,periodos
0,48.01.002,BIOMEDICINA,3184,8
1,34.02.003,ENFERMAGEM,4145,10
2,15.05.003,FARMÁCIA,4317,10
3,16.01.004,MEDICINA,7001,12
4,18.01.004,MEDICINA VETERINÁRIA,5170,10
5,17.01.003,ODONTOLOGIA,4035,9


In [13]:
# nós: Disciplina (sem duplicadas, uma linha por código)
nos_disciplina = (df.rename(columns={"chtotal": "chTotal"})
                  [["codigo", "nome", "nomeBase", "cht", "chp", "ches", "chTotal", "chex", "desativada"]]
                  .drop_duplicates("codigo"))

# confere se a mesma disciplina não veio com dados diferentes em cursos diferentes
conflito = df.groupby("codigo")[["nome", "chtotal"]].nunique()
print("disciplinas:", len(nos_disciplina), "| conflitos:", int((conflito > 1).any(axis=1).sum()))

disciplinas: 356 | conflitos: 0


In [14]:
# nós: Departamento
nos_depto = pd.DataFrame({"sigla": sorted(df["departamento"].unique())})
len(nos_depto)

42

In [15]:
# relação: OFERECE (curso -> disciplina)
rel_oferece = df[["curso", "codigo", "periodo", "tipo", "chprereq", "periodosCompletos"]].rename(
    columns={"chprereq": "chMinima"})
len(rel_oferece)

357

In [16]:
# relação: REQUER (disciplina -> pré ou co-requisito)
pre = df[["codigo", "preCodigos"]].explode("preCodigos").dropna().rename(columns={"preCodigos": "requisito"})
pre["tipo"] = "pre"
co = df[["codigo", "coCodigos"]].explode("coCodigos").dropna().rename(columns={"coCodigos": "requisito"})
co["tipo"] = "co"
rel_requer = pd.concat([pre, co]).drop_duplicates()

# requisito que não existe como disciplina
orfaos = rel_requer[~rel_requer["requisito"].isin(nos_disciplina["codigo"])]
print("requer:", len(rel_requer), "| órfãos:", len(orfaos))

requer: 614 | órfãos: 0


In [17]:
# relação: DO_DEPTO
rel_depto = df[["codigo", "departamento"]].drop_duplicates()

In [18]:
# relação: EQUIVALE_A (opcional, lê data/equivalencias.csv se existir)
# formato: codigo_a,codigo_b
arq_eq = DATA_DIR / "equivalencias.csv"
if arq_eq.exists():
    rel_equivale = pd.read_csv(arq_eq, dtype=str)
    rel_equivale = rel_equivale.apply(lambda c: c.str.strip().str.upper())
    rel_equivale = rel_equivale[rel_equivale["codigo_a"] != rel_equivale["codigo_b"]].drop_duplicates()
    print("equivalências:", len(rel_equivale))
else:
    rel_equivale = pd.DataFrame(columns=["codigo_a", "codigo_b"])
    print("sem equivalencias.csv, pulando")

equivalências: 0


In [19]:
# sugestão de equivalências pro grupo revisar: mesmo nomeBase em cursos diferentes
# (não é carregado, só salvo em data/processed)
pares = df[["curso", "codigo", "nome", "nomeBase"]].merge(df[["curso", "codigo", "nome", "nomeBase"]], on="nomeBase")
pares = pares[(pares["curso_x"] != pares["curso_y"]) & (pares["codigo_x"] < pares["codigo_y"])]
sugestao = pares.rename(columns={"codigo_x": "codigo_a", "codigo_y": "codigo_b", "nome_x": "nome_a", "nome_y": "nome_b"})
sugestao = sugestao[["codigo_a", "nome_a", "codigo_b", "nome_b"]].drop_duplicates()
sugestao.to_csv(OUT_DIR / "equivalencias_sugeridas.csv", index=False)
print("pares sugeridos:", len(sugestao))

pares sugeridos: 108


In [20]:
# salva as tabelas tratadas
for nome, tab in {"cursos": nos_curso, "disciplinas": nos_disciplina, "departamentos": nos_depto,
                  "oferece": rel_oferece, "requer": rel_requer, "do_depto": rel_depto}.items():
    tab.to_csv(OUT_DIR / f"{nome}.csv", index=False)

In [21]:
# NaN -> None antes de mandar pro neo4j
def registros(tab):
    return tab.astype(object).where(tab.notna(), None).to_dict("records")

## 3. Load
1. **Schema:** executar `cypher/schema.cypher` (constraints e índices).
2. **Dados:** `MERGE` é idempotente (slide 30); valores sempre por parâmetro, nunca por concatenação (slide 42); `UNWIND` em lotes.

In [22]:
def executar_arquivo_cypher(caminho):
    texto = re.sub(r"//.*", "", Path(caminho).read_text(encoding="utf-8"))
    comandos = [c.strip() for c in texto.split(";") if c.strip()]
    for c in comandos:
        driver.execute_query(c, database_=NEO4J_DB)
    return len(comandos)

executar_arquivo_cypher(CYPHER_DIR / "schema.cypher")

5

In [23]:
# carga em lotes
def carregar(query, tab, lote=500):
    rows = registros(tab)
    for i in range(0, len(rows), lote):
        driver.execute_query(query, rows=rows[i:i + lote], database_=NEO4J_DB)
    return len(rows)

In [24]:
# nós
carregar("""
UNWIND $rows AS r
MERGE (c:Curso {codigo: r.codigo})
SET c.nome = r.nome, c.chObrigatoria = r.chObrigatoria, c.periodos = r.periodos
""", nos_curso)

carregar("""
UNWIND $rows AS r
MERGE (d:Disciplina {codigo: r.codigo})
SET d.nome = r.nome, d.nomeBase = r.nomeBase, d.cht = r.cht, d.chp = r.chp,
    d.ches = r.ches, d.chTotal = r.chTotal, d.chex = r.chex, d.desativada = r.desativada
""", nos_disciplina)

carregar("""
UNWIND $rows AS r
MERGE (:Departamento {sigla: r.sigla})
""", nos_depto)

42

In [25]:
# relações
carregar("""
UNWIND $rows AS r
MATCH (c:Curso {codigo: r.curso})
MATCH (d:Disciplina {codigo: r.codigo})
MERGE (c)-[o:OFERECE]->(d)
SET o.periodo = r.periodo, o.tipo = r.tipo, o.chMinima = r.chMinima, o.periodosCompletos = r.periodosCompletos
""", rel_oferece)

carregar("""
UNWIND $rows AS r
MATCH (d:Disciplina {codigo: r.codigo})
MATCH (req:Disciplina {codigo: r.requisito})
MERGE (d)-[:REQUER {tipo: r.tipo}]->(req)
""", rel_requer)

carregar("""
UNWIND $rows AS r
MATCH (d:Disciplina {codigo: r.codigo})
MATCH (dp:Departamento {sigla: r.departamento})
MERGE (d)-[:DO_DEPTO]->(dp)
""", rel_depto)

carregar("""
UNWIND $rows AS r
MATCH (a:Disciplina {codigo: r.codigo_a})
MATCH (b:Disciplina {codigo: r.codigo_b})
MERGE (a)-[:EQUIVALE_A]->(b)
""", rel_equivale)

0

## 4. Validação

In [26]:
def consulta(q, **params):
    recs, _, _ = driver.execute_query(q, params, database_=NEO4J_DB)
    return pd.DataFrame([r.data() for r in recs])

In [27]:
# 1. nós por label
consulta("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS total ORDER BY label")

,label,total
0,Curso,6
1,Departamento,42
2,Disciplina,356


In [28]:
# 2. relações por tipo
consulta("MATCH ()-[r]->() RETURN type(r) AS relacao, count(*) AS total ORDER BY relacao")

,relacao,total
0,DO_DEPTO,356
1,OFERECE,357
2,REQUER,614


In [29]:
# 3. disciplinas por curso: pdf x grafo
grafo = consulta("MATCH (c:Curso)-[:OFERECE]->(d) RETURN c.codigo AS curso, count(d) AS grafo")
pdf_qtd = df.groupby("curso").size().rename("pdf").reset_index()
v3 = pdf_qtd.merge(grafo, on="curso")
v3["ok"] = v3["pdf"] == v3["grafo"]
v3

,curso,pdf,grafo,ok
0,15.05.003,74,74,True
1,16.01.004,65,65,True
2,17.01.003,62,62,True
3,18.01.004,70,70,True
4,34.02.003,53,53,True
5,48.01.002,33,33,True


In [30]:
# 4. carga horária somada x carga obrigatória impressa no pdf
consulta("""
MATCH (c:Curso)-[:OFERECE]->(d:Disciplina)
WITH c, sum(d.chTotal) AS soma
RETURN c.nome AS curso, c.chObrigatoria AS pdf, soma AS grafo, soma = c.chObrigatoria AS ok
ORDER BY curso
""")

,curso,pdf,grafo,ok
0,BIOMEDICINA,3184,3184,True
1,ENFERMAGEM,4145,4145,True
2,FARMÁCIA,4317,4317,True
3,MEDICINA,7001,7001,True
4,MEDICINA VETERINÁRIA,5170,5170,True
5,ODONTOLOGIA,4035,4035,True


In [31]:
# 5. nenhum código duplicado
consulta("""
MATCH (d:Disciplina)
WITH d.codigo AS codigo, count(*) AS n
WHERE n > 1
RETURN count(*) AS duplicados
""")

,duplicados
0,0


In [32]:
# 6. nenhum requisito perdido: requer no grafo = requer no pandas
r = consulta("MATCH ()-[r:REQUER]->() RETURN count(r) AS grafo")
print("pandas:", len(rel_requer), "| grafo:", int(r["grafo"][0]), "| órfãos:", len(orfaos))

pandas: 614 | grafo: 614 | órfãos: 0


In [33]:
# 7. nenhum campo vazio onde não pode
consulta("""
MATCH (d:Disciplina)
WHERE d.nome IS NULL OR d.nome = '' OR d.chTotal IS NULL OR NOT (d)-[:DO_DEPTO]->()
RETURN count(d) AS problemas
""")

,problemas
0,0


In [34]:
driver.close()